# fall-classifier-bench — análise de runs

Lê todos os CSVs em `output/csv/` e gera comparativos entre execuções.

**Workflow esperado:**
1. Rodar `python main.py --tag X` várias vezes na Raspi com diferentes parâmetros
2. `scp -r vigia@raspi:~/fall-classifier-bench/output/csv ./output/csv/`
3. Rodar este notebook

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

CSV_DIR = Path('output/csv')

runs = {}
for csv in sorted(CSV_DIR.glob('*.csv')):
    # nome: YYYYMMDD_HHMMSS_<tag>.csv
    stem = csv.stem
    tag = '_'.join(stem.split('_')[2:]) or stem
    df = pd.read_csv(csv)
    df['run_id'] = stem
    df['tag'] = tag
    runs[stem] = df
    print(f'{stem}: {len(df)} linhas, tag={tag}')

if not runs:
    print('Nenhum CSV em', CSV_DIR.resolve())
else:
    all_df = pd.concat(runs.values(), ignore_index=True)
    print('\nTotal:', len(all_df), 'linhas em', len(runs), 'runs')

## Tabela comparativa entre runs

In [ ]:
summary = all_df.groupby('tag').agg(
    n_inferencias=('frame_idx', 'count'),
    t_yolo_ms_mean=('t_yolo_ms', 'mean'),
    t_yolo_ms_p95=('t_yolo_ms', lambda s: s.quantile(0.95)),
    t_gru_ms_mean=('t_gru_ms', 'mean'),
    n_valid_kpts_mean=('n_valid_kpts', 'mean'),
    pct_invalid=('label', lambda s: (s == 'INVALID').mean() * 100),
    pct_fall=('label', lambda s: (s == 'FALL').mean() * 100),
    pct_alert=('alert', lambda s: s.mean() * 100),
).round(2)
summary

## Latência YOLO por run

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for tag, group in all_df.groupby('tag'):
    ax.hist(group['t_yolo_ms'], bins=40, alpha=0.5, label=tag)
ax.set_xlabel('t_yolo (ms)')
ax.set_ylabel('frames')
ax.set_title('Distribuição de latência YOLO por run')
ax.legend()
plt.tight_layout(); plt.show()

## Qualidade dos keypoints — quantos foram válidos por janela

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for tag, group in all_df.groupby('tag'):
    ax.hist(group['n_valid_kpts'], bins=18, alpha=0.5, label=tag, range=(0, 17))
ax.set_xlabel('n_valid_kpts (de 17 possíveis)')
ax.set_ylabel('inferências')
ax.set_title('Quantos keypoints sobreviveram ao threshold por inferência')
ax.legend()
plt.tight_layout(); plt.show()

## Probabilidade de FALL ao longo do tempo

In [ ]:
fig, axes = plt.subplots(len(runs), 1, figsize=(12, 3 * len(runs)), sharex=False)
if len(runs) == 1:
    axes = [axes]
for ax, (run_id, df) in zip(axes, runs.items()):
    ax.plot(df['frame_idx'], df['prob_fall'], label='prob_fall', linewidth=0.8)
    fall_alert = df[df['alert'] == 1]
    if not fall_alert.empty:
        ax.scatter(fall_alert['frame_idx'], fall_alert['prob_fall'],
                   c='red', s=20, label='alert confirmado', zorder=5)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.5)
    ax.set_ylabel('P(FALL)')
    ax.set_title(run_id, fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylim(0, 1)
axes[-1].set_xlabel('frame_idx')
plt.tight_layout(); plt.show()

## Foco em janelas com keypoints bons

Filtra apenas inferências onde a maioria dos keypoints obrigatórios estava presente.
Isolar a qualidade da GRU eliminando casos em que YOLO falhou.

In [ ]:
MIN_VALID_FRAMES = 15  # 75% da janela de 20 frames

clean = all_df[all_df['n_valid_frames_window'] >= MIN_VALID_FRAMES].copy()
print(f'Inferências com >= {MIN_VALID_FRAMES} frames válidos: {len(clean)}/{len(all_df)}')

clean_summary = clean.groupby('tag').agg(
    n=('frame_idx', 'count'),
    pct_fall=('label', lambda s: (s == 'FALL').mean() * 100),
    pct_alert=('alert', lambda s: s.mean() * 100),
    prob_fall_mean=('prob_fall', 'mean'),
).round(2)
clean_summary